In [1]:
import os, subprocess

!pip install -q opencv-python-headless python_speech_features scipy tqdm numpy
!pip install -q "scenedetect[opencv]>=0.6.3"

!rm -rf /kaggle/working/syncnet_python
!git clone https://github.com/joonson/syncnet_python.git /kaggle/working/syncnet_python

SYNCNET_DIR = "/kaggle/working/syncnet_python"
assert os.path.exists(f"{SYNCNET_DIR}/run_pipeline.py"), "Clone failed."

os.makedirs(f"{SYNCNET_DIR}/data", exist_ok=True)
os.makedirs(f"{SYNCNET_DIR}/detectors/s3fd/weights", exist_ok=True)

!wget -q https://www.robots.ox.ac.uk/~vgg/software/lipsync/data/syncnet_v2.model \
    -O {SYNCNET_DIR}/data/syncnet_v2.model
!wget -q https://www.robots.ox.ac.uk/~vgg/software/lipsync/data/sfd_face.pth \
    -O {SYNCNET_DIR}/detectors/s3fd/weights/sfd_face.pth

for w in [f"{SYNCNET_DIR}/data/syncnet_v2.model",
          f"{SYNCNET_DIR}/detectors/s3fd/weights/sfd_face.pth"]:
    sz = os.path.getsize(w) if os.path.exists(w) else 0
    print(f"{w.split('/')[-1]}: {sz/1e6:.1f} MB")
    assert sz > 1_000_000, f"{w} download failed."

!nvidia-smi --query-gpu=index,name,memory.total --format=csv
print("Setup complete.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.2/146.2 kB 8.2 MB/s eta 0:00:00
Cloning into '/kaggle/working/syncnet_python'...
remote: Enumerating objects: 155, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 155 (delta 77), reused 75 (delta 67), pack-reused 55 (from 1)
Receiving objects: 100% (155/155), 106.12 KiB | 7.58 MiB/s, done.
Resolving deltas: 100% (96/96), done.
syncnet_v2.model: 54.6 MB
sfd_face.pth: 89.8 MB
index, name, memory.total [MiB]
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB
Setup complete.


In [5]:
import os, glob, math, json, subprocess, shutil, sys, time, threading
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count()
N_CPUS = os.cpu_count()
print(f"GPUs: {N_GPUS}   CPU cores: {N_CPUS}")

SYNCNET_DIR = "/kaggle/working/syncnet_python"
WORK_DIR    = "/kaggle/working"

# ---------------- dataset root ----------------
DATA_ROOT = "/kaggle/input/datasets/mustakimbinmohsin/speech/dataset"
if not os.path.isdir(DATA_ROOT):
    # fall back to a broad search if the exact path shifts
    hits = [p for p in glob.glob("/kaggle/input/**/dataset", recursive=True)
            if os.path.isdir(os.path.join(p, "train"))]
    assert hits, "Dataset root not found under /kaggle/input"
    DATA_ROOT = hits[0]
print("DATA_ROOT:", DATA_ROOT)

VIDEO_EXTS = (".mpg", ".mp4", ".avi", ".mov", ".mkv")

# ---------------- detect directory layout ----------------
# A: <root>/<split>/video/<speaker>/*.mpg
# B: <root>/<split>/<speaker>/video/*.mpg
def _count(pattern):
    return len([f for f in glob.glob(pattern) if f.lower().endswith(VIDEO_EXTS)])

n_A = _count(os.path.join(DATA_ROOT, "train", "video", "*", "*"))
n_B = _count(os.path.join(DATA_ROOT, "train", "*", "video", "*"))
LAYOUT = "A" if n_A >= n_B else "B"
print(f"Layout A (split/video/spk): {n_A} files | Layout B (split/spk/video): {n_B} files")
print(f"--> using layout {LAYOUT}")
assert max(n_A, n_B) > 0, "No video files found in either layout."

def video_dir(split, speaker):
    """Directory holding a speaker's videos for a split."""
    return (os.path.join(DATA_ROOT, split, "video", speaker) if LAYOUT == "A"
            else os.path.join(DATA_ROOT, split, speaker, "video"))

def list_videos(split, speaker):
    d = video_dir(split, speaker)
    if not os.path.isdir(d):
        return []
    return sorted(f for f in os.listdir(d) if f.lower().endswith(VIDEO_EXTS))

# ---------------- speakers ----------------
if LAYOUT == "A":
    spk_src = os.path.join(DATA_ROOT, "train", "video")
else:
    spk_src = os.path.join(DATA_ROOT, "train")
SPEAKERS = sorted(
    [d for d in os.listdir(spk_src) if os.path.isdir(os.path.join(spk_src, d))
     and d not in ("video", "audio")],
    key=lambda s: int("".join(ch for ch in s if ch.isdigit()) or 0)
)
print("Speakers:", SPEAKERS)

for split in ["train", "test"]:
    counts = {s: len(list_videos(split, s)) for s in SPEAKERS}
    print(f"  {split}: {sum(counts.values())} videos  {counts}")

# ---------------- manifest (informational) ----------------
for mp in [os.path.join(DATA_ROOT, "manifest.csv"),
           os.path.join(os.path.dirname(DATA_ROOT), "manifest.csv")]:
    if os.path.exists(mp):
        df = pd.read_csv(mp)
        print(f"\nmanifest.csv: {len(df)} rows | columns: {list(df.columns)}")
        print(df.head(3).to_string())
        break

# ---------------- output dirs ----------------
TRAIN_PROC_DIR = f"{WORK_DIR}/proc_train"
TEST_PROC_DIR  = f"{WORK_DIR}/proc_test"
TRAIN_FEAT_DIR = f"{WORK_DIR}/features/train"
TEST_FEAT_DIR  = f"{WORK_DIR}/features/test"
CKPT_DIR       = f"{WORK_DIR}/checkpoints"
for d in [TRAIN_PROC_DIR, TEST_PROC_DIR, TRAIN_FEAT_DIR, TEST_FEAT_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

# ---------------- paper hyperparameters (Sec. 4.1) ----------------
TAU      = 15            # max delay offset -> distribution length 2*tau+1
DIST_LEN = 2 * TAU + 1   # 31
FPS, SR  = 25, 16000
AR_BLOCKS, AR_HEADS, AR_DIM, DROPOUT = 2, 16, 256, 0.1
LR, WEIGHT_DECAY = 1e-3, 1e-6
BATCH_SIZE   = 16 * max(1, N_GPUS)
EPOCHS       = 40
WARMUP_STEPS = 300
SEQ_LEN      = None      # set in Cell 7 from actual feature lengths

print(f"\nDIST_LEN={DIST_LEN}  batch={BATCH_SIZE}  epochs={EPOCHS}")

GPUs: 2   CPU cores: 4
DATA_ROOT: /kaggle/input/datasets/mustakimbinmohsin/speech/dataset
Layout A (split/video/spk): 6400 files | Layout B (split/spk/video): 0 files
--> using layout A
Speakers: ['s1', 's2', 's3', 's4', 's5', 's6', 's7', 's8']
  train: 6400 videos  {'s1': 800, 's2': 800, 's3': 800, 's4': 800, 's5': 800, 's6': 800, 's7': 800, 's8': 800}
  test: 1600 videos  {'s1': 200, 's2': 200, 's3': 200, 's4': 200, 's5': 200, 's6': 200, 's7': 200, 's8': 200}

manifest.csv: 8000 rows | columns: ['speaker', 'utterance_id', 'audio_path', 'video_path', 'split']
  speaker utterance_id                                                   audio_path                                                   video_path  split
0      s1       sbia1s  D:\Audio+video\gridcorpus\dataset\train\audio\s1\sbia1s.wav  D:\Audio+video\gridcorpus\dataset\train\video\s1\sbia1s.mpg  train
1      s1       pbbc7a  D:\Audio+video\gridcorpus\dataset\train\audio\s1\pbbc7a.wav  D:\Audio+video\gridcorpus\dataset\train\vide

In [4]:
preprocess_worker_code = r'''
import sys, os, subprocess, time, shutil

speaker_list = sys.argv[1].split(",")
video_root   = os.path.abspath(sys.argv[2])   # <root>/<split>/video
out_data_dir = os.path.abspath(sys.argv[3])
gpu_id       = sys.argv[4]
SYNCNET_DIR  = "/kaggle/working/syncnet_python"

os.environ["CUDA_VISIBLE_DEVICES"] = gpu_id
import torch
print(f"[GPU {gpu_id}] cuda={torch.cuda.is_available()}", flush=True)

os.makedirs(out_data_dir, exist_ok=True)

jobs = []
for spk in speaker_list:
    vdir = os.path.join(video_root, spk)
    if os.path.isdir(vdir):
        for f in sorted(os.listdir(vdir)):
            if f.lower().endswith((".mpg", ".mp4", ".avi")):
                jobs.append((spk, f))

total = len(jobs); n_ok = n_fail = 0; shown = False; t0 = time.time()
print(f"[GPU {gpu_id}] starting: {total} videos", flush=True)

for idx, (spk, vf) in enumerate(jobs, 1):
    vpath = os.path.join(video_root, spk, vf)
    ref   = f"{spk}_{os.path.splitext(vf)[0]}"
    check = os.path.join(out_data_dir, "pycrop", ref)

    if glob_has := (os.path.isdir(check) and
                    any(x.endswith(".avi") for x in os.listdir(check))):
        n_ok += 1
        continue

    r = subprocess.run(
        ["python", "run_pipeline.py",
         "--videofile", os.path.abspath(vpath),
         "--reference", ref,
         "--data_dir", out_data_dir,
         "--min_track", "25",        # GRID clips are ~75 frames; default 100 rejects all
         "--min_face_size", "40",    # 360x288 source -> faces below default 100px
         "--overwrite"],
        cwd=SYNCNET_DIR, check=False, capture_output=True, text=True)

    if os.path.isdir(check) and any(x.endswith(".avi") for x in os.listdir(check)):
        n_ok += 1
        for sub in ["pyframes", "pyavi", "pywork"]:   # free disk immediately
            shutil.rmtree(os.path.join(out_data_dir, sub, ref), ignore_errors=True)
    else:
        n_fail += 1
        if not shown:
            sys.stderr.write(f"\n[GPU {gpu_id}] FIRST FAILURE: {vpath}\n")
            sys.stderr.write(f"STDERR tail:\n{r.stderr[-500:]}\n")
            sys.stderr.flush(); shown = True
        if n_fail >= 15 and n_ok == 0:
            sys.stderr.write(f"[GPU {gpu_id}] ABORT: 15 failures, 0 successes.\n")
            sys.stderr.flush(); break

    if idx % 25 == 0:
        el = time.time() - t0
        eta = (el / idx) * (total - idx) / 60
        print(f"[GPU {gpu_id}] {idx}/{total} | ok={n_ok} fail={n_fail} "
              f"| {el/60:.1f}m | ETA ~{eta:.1f}m", flush=True)

print(f"[GPU {gpu_id}] done: {n_ok} ok, {n_fail} fail, {(time.time()-t0)/60:.1f} min",
      flush=True)
'''

extract_worker_code = r'''
import sys, os, glob, shutil, time, traceback
import numpy as np
import torch

speaker_list  = sys.argv[1].split(",")
proc_data_dir = os.path.abspath(sys.argv[2])
out_feat_dir  = os.path.abspath(sys.argv[3])
gpu_id        = sys.argv[4]
tau           = int(sys.argv[5])
SYNCNET_DIR   = "/kaggle/working/syncnet_python"

os.environ["CUDA_VISIBLE_DEVICES"] = gpu_id
sys.path.insert(0, SYNCNET_DIR)
from SyncNetInstance import SyncNetInstance

class Opt:
    """evaluate() reads tmp_dir, reference, batch_size, vshift."""
    def __init__(self, tmp_dir, reference, vshift):
        self.tmp_dir = tmp_dir
        self.reference = reference
        self.batch_size = 20
        self.vshift = vshift

net = SyncNetInstance()
net.loadParameters(os.path.join(SYNCNET_DIR, "data", "syncnet_v2.model"))
print(f"[GPU {gpu_id}] SyncNet loaded", flush=True)

os.makedirs(out_feat_dir, exist_ok=True)
pycrop = os.path.join(proc_data_dir, "pycrop")
if not os.path.isdir(pycrop):
    print(f"[GPU {gpu_id}] no pycrop at {pycrop}", flush=True); sys.exit(0)

mine = [t for t in sorted(os.listdir(pycrop)) if t.split("_")[0] in speaker_list]
total = len(mine); n_ok = n_skip = n_fail = 0; shown = False; t0 = time.time()
print(f"[GPU {gpu_id}] starting: {total} tracks", flush=True)

for idx, track in enumerate(mine, 1):
    try:
        out_path = os.path.join(out_feat_dir, f"{track}.npy")
        if os.path.exists(out_path):
            n_skip += 1; continue

        avis = sorted(glob.glob(os.path.join(pycrop, track, "*.avi")))
        if not avis:
            n_fail += 1; continue

        tmp = os.path.join("/kaggle/working/synctmp", f"g{gpu_id}")
        os.makedirs(tmp, exist_ok=True)

        _, _, dists = net.evaluate(Opt(tmp, track, tau),
                                   videofile=os.path.abspath(avis[0]))

        # distances -> similarity -> softmax over the 2*tau+1 window = S(i,j), Eq.(1)
        probs = torch.softmax(torch.tensor(-np.stack(dists)), dim=-1).numpy()
        np.save(out_path, probs.astype(np.float32))
        shutil.rmtree(os.path.join(tmp, track), ignore_errors=True)
        n_ok += 1

    except Exception:
        n_fail += 1
        if not shown:
            sys.stderr.write(f"\n[GPU {gpu_id}] FIRST FAILURE track={track}\n")
            sys.stderr.write(traceback.format_exc()[-800:])
            sys.stderr.flush(); shown = True
        if n_fail >= 10 and n_ok == 0:
            sys.stderr.write(f"[GPU {gpu_id}] ABORT: 10 failures, 0 successes.\n")
            sys.stderr.flush(); break

    if idx % 50 == 0:
        el = time.time() - t0
        eta = (el / idx) * (total - idx) / 60
        print(f"[GPU {gpu_id}] {idx}/{total} | ok={n_ok} skip={n_skip} fail={n_fail} "
              f"| {el/60:.1f}m | ETA ~{eta:.1f}m", flush=True)

print(f"[GPU {gpu_id}] done: {n_ok} new, {n_skip} skip, {n_fail} fail, "
      f"{(time.time()-t0)/60:.1f} min", flush=True)
'''

with open("preprocess_worker.py", "w") as f: f.write(preprocess_worker_code)
with open("extract_worker.py", "w") as f: f.write(extract_worker_code)

for s in ["preprocess_worker.py", "extract_worker.py"]:
    r = subprocess.run(["python", "-m", "py_compile", s], capture_output=True, text=True)
    print(f"{s}: {'OK' if r.returncode == 0 else 'SYNTAX ERROR' + r.stderr}")

preprocess_worker.py: OK
extract_worker.py: OK


In [2]:
extract_worker_code = r'''
import sys, os, glob, shutil, time, traceback, contextlib, io
import numpy as np
import torch

speaker_list  = sys.argv[1].split(",")
proc_data_dir = os.path.abspath(sys.argv[2])
out_feat_dir  = os.path.abspath(sys.argv[3])
gpu_id        = sys.argv[4]
tau           = int(sys.argv[5])
SYNCNET_DIR   = "/kaggle/working/syncnet_python"

os.environ["CUDA_VISIBLE_DEVICES"] = gpu_id
sys.path.insert(0, SYNCNET_DIR)
from SyncNetInstance import SyncNetInstance

class Opt:
    """evaluate() reads tmp_dir, reference, batch_size, vshift."""
    def __init__(self, tmp_dir, reference, vshift):
        self.tmp_dir = tmp_dir
        self.reference = reference
        self.batch_size = 20
        self.vshift = vshift

net = SyncNetInstance()
net.loadParameters(os.path.join(SYNCNET_DIR, "data", "syncnet_v2.model"))
print(f"[GPU {gpu_id}] SyncNet loaded", flush=True)

os.makedirs(out_feat_dir, exist_ok=True)
pycrop = os.path.join(proc_data_dir, "pycrop")
if not os.path.isdir(pycrop):
    print(f"[GPU {gpu_id}] no pycrop at {pycrop}", flush=True); sys.exit(0)

mine = [t for t in sorted(os.listdir(pycrop)) if t.split("_")[0] in speaker_list]
total = len(mine); n_ok = n_skip = n_fail = 0; shown = False; t0 = time.time()
print(f"[GPU {gpu_id}] starting: {total} tracks", flush=True)

for idx, track in enumerate(mine, 1):
    try:
        out_path = os.path.join(out_feat_dir, f"{track}.npy")
        if os.path.exists(out_path):
            n_skip += 1; continue

        avis = sorted(glob.glob(os.path.join(pycrop, track, "*.avi")))
        if not avis:
            n_fail += 1; continue

        # tmp_dir is WRITABLE working space -- proc_data_dir (pycrop source)
        # is read-only when resumed from a committed checkpoint, so this
        # must NOT point back into proc_data_dir.
        tmp = os.path.join("/kaggle/working/synctmp", f"g{gpu_id}")
        os.makedirs(tmp, exist_ok=True)

        # evaluate() internally prints a benign "Audio/video lengths are
        # different" notice on nearly every GRID clip (fixed ~36ms encoding
        # mismatch, harmless). Suppress it so heartbeat logs stay readable.
        with contextlib.redirect_stdout(io.StringIO()):
            _, _, dists = net.evaluate(Opt(tmp, track, tau),
                                       videofile=os.path.abspath(avis[0]))

        # distances -> similarity -> softmax over the 2*tau+1 window = S(i,j), Eq.(1)
        probs = torch.softmax(torch.tensor(-np.stack(dists)), dim=-1).numpy()
        np.save(out_path, probs.astype(np.float32))
        shutil.rmtree(os.path.join(tmp, track), ignore_errors=True)
        n_ok += 1

    except Exception:
        n_fail += 1
        if not shown:
            sys.stderr.write(f"\n[GPU {gpu_id}] FIRST FAILURE track={track}\n")
            sys.stderr.write(traceback.format_exc()[-800:])
            sys.stderr.flush(); shown = True
        if n_fail >= 10 and n_ok == 0:
            sys.stderr.write(f"[GPU {gpu_id}] ABORT: 10 failures, 0 successes.\n")
            sys.stderr.flush(); break

    if idx % 50 == 0:
        el = time.time() - t0
        eta = (el / idx) * (total - idx) / 60
        print(f"[GPU {gpu_id}] {idx}/{total} | ok={n_ok} skip={n_skip} fail={n_fail} "
              f"| {el/60:.1f}m | ETA ~{eta:.1f}m", flush=True)

print(f"[GPU {gpu_id}] done: {n_ok} new, {n_skip} skip, {n_fail} fail, "
      f"{(time.time()-t0)/60:.1f} min", flush=True)
'''

with open("extract_worker.py", "w") as f:
    f.write(extract_worker_code)

r = subprocess.run(["python", "-m", "py_compile", "extract_worker.py"],
                    capture_output=True, text=True)
print(f"extract_worker.py: {'OK' if r.returncode == 0 else 'SYNTAX ERROR' + r.stderr}")

extract_worker.py: OK


In [ ]:
# Verifies ONE video end-to-end, and ONE SyncNet feature extraction,
# before committing hours of compute.
vids = sorted(glob.glob(os.path.join(DATA_ROOT, "train", "video", "*", "*.mpg")))
assert vids, "No videos found."
print("Testing:", vids[0])

shutil.rmtree("/kaggle/working/debug_proc", ignore_errors=True)
r = subprocess.run(
    ["python", "run_pipeline.py",
     "--videofile", os.path.abspath(vids[0]),
     "--reference", "debug_test",
     "--data_dir", "/kaggle/working/debug_proc",
     "--min_track", "25", "--min_face_size", "40", "--overwrite"],
    cwd=SYNCNET_DIR, capture_output=True, text=True)

print("run_pipeline rc:", r.returncode)
crops = glob.glob("/kaggle/working/debug_proc/pycrop/debug_test/*.avi")
print("crops:", crops)
if not crops:
    print("STDERR tail:\n", r.stderr[-2000:])
assert crops, "STEP 1 FAILED -- face-track cropping produced nothing."

# --- step 2: SyncNet feature extraction on that crop ---
sys.path.insert(0, SYNCNET_DIR)
from SyncNetInstance import SyncNetInstance

class Opt:
    def __init__(self, tmp_dir, reference, vshift):
        self.tmp_dir, self.reference = tmp_dir, reference
        self.batch_size, self.vshift = 20, vshift

shutil.rmtree("/kaggle/working/debug_tmp", ignore_errors=True)
os.makedirs("/kaggle/working/debug_tmp", exist_ok=True)

net = SyncNetInstance()
net.loadParameters(f"{SYNCNET_DIR}/data/syncnet_v2.model")
offset, conf, dists = net.evaluate(
    Opt("/kaggle/working/debug_tmp", "debug_test", TAU),
    videofile=os.path.abspath(crops[0]))

d = np.stack(dists)
probs = torch.softmax(torch.tensor(-d), dim=-1).numpy()
print(f"\noffset={offset}  conf={conf:.3f}")
print(f"feature shape: {probs.shape}  (expected [T, {DIST_LEN}])")
print(f"row sums (should be ~1.0): {probs[:3].sum(axis=1)}")
assert probs.shape[1] == DIST_LEN, f"Got width {probs.shape[1]}, expected {DIST_LEN}"
print("\nDIAGNOSTIC PASSED -- proceed to Cell 5.")

In [ ]:
WORKERS_PER_GPU = 2
NUM_WORKERS = WORKERS_PER_GPU * max(1, N_GPUS)

def chunk_list(lst, n):
    k, m = divmod(len(lst), n)
    return [lst[i*k+min(i,m):(i+1)*k+min(i+1,m)] for i in range(n)]

def launch_parallel(script, in_dir, out_dir, extra_args=None):
    """Launches all workers at once; GPUs assigned round-robin."""
    extra_args = extra_args or []
    chunks = [c for c in chunk_list(SPEAKERS, NUM_WORKERS) if c]
    procs = []
    for i, chunk in enumerate(chunks):
        gpu = str(i % max(1, N_GPUS))
        print(f"  worker {i} -> GPU {gpu}: {chunk}", flush=True)
        procs.append(subprocess.Popen(
            ["python", "-u", script, ",".join(chunk),
             os.path.abspath(in_dir), os.path.abspath(out_dir), gpu] + extra_args))
    codes = [p.wait() for p in procs]
    for i, c in enumerate(codes):
        if c != 0:
            print(f"  [warn] worker {i} exit code {c}")

def run_monitored(stage, script, in_dir, out_dir, extra_args, count_glob, target):
    """Background monitor counts real output files -- immune to stdout deadlocks."""
    stop = threading.Event()
    def mon():
        t = time.time(); n0 = len(glob.glob(count_glob)); stall = 0; last = n0
        print(f"[monitor] {stage}: from {n0}/{target}", flush=True)
        while not stop.wait(30):
            n = len(glob.glob(count_glob)); el = time.time()-t; made = n-n0
            if made > 0:
                rate = made/el
                print(f"[monitor] {stage}: {n}/{target} | {rate*60:.0f}/min "
                      f"| {el/60:.1f}m | ETA ~{(target-n)/rate/60:.1f}m", flush=True)
            else:
                print(f"[monitor] {stage}: {n}/{target} | {el/60:.1f}m | no output yet",
                      flush=True)
            stall = stall+1 if n == last else 0
            if stall >= 6:
                print("[monitor] WARNING: no progress in 3 min.", flush=True); stall = 0
            last = n
    th = threading.Thread(target=mon, daemon=True); th.start()
    t0 = time.time()
    try:
        launch_parallel(script, in_dir, out_dir, extra_args)
    finally:
        stop.set(); th.join(timeout=2)
    print(f"{stage}: {(time.time()-t0)/60:.1f} min\n", flush=True)


for split, proc_dir in [("train", TRAIN_PROC_DIR), ("test", TEST_PROC_DIR)]:
    video_root = os.path.join(DATA_ROOT, split, "video")
    target = len(glob.glob(os.path.join(video_root, "*", "*.mpg")))
    print(f"=== Preprocessing {split.upper()} ({target} videos) ===")
    run_monitored(split.upper(), "preprocess_worker.py", video_root, proc_dir, None,
                  os.path.join(proc_dir, "pycrop", "*", "*.avi"), target)

for d, name in [(TRAIN_PROC_DIR, "TRAIN"), (TEST_PROC_DIR, "TEST")]:
    n = len(glob.glob(os.path.join(d, "pycrop", "*", "*.avi")))
    print(f"{name}: {n} usable face tracks")

In [3]:
import os, sys, glob, pickle, shutil, time, threading, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count()
print(f"Device: {DEVICE}  GPUs: {N_GPUS}")

WORK_DIR    = "/kaggle/working"
SYNCNET_DIR = "/kaggle/working/syncnet_python"

TRAIN_PROC_DIR = f"{WORK_DIR}/proc_train"
TEST_PROC_DIR  = f"{WORK_DIR}/proc_test"
TRAIN_FEAT_DIR = f"{WORK_DIR}/features/train"
TEST_FEAT_DIR  = f"{WORK_DIR}/features/test"
CKPT_DIR       = f"{WORK_DIR}/checkpoints"

for d in [TRAIN_FEAT_DIR, TEST_FEAT_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

# Paper hyperparameters (Sec. 4.1)
TAU          = 15
DIST_LEN     = 2 * TAU + 1  # 31
FPS, SR      = 25, 16000
AR_BLOCKS    = 2
AR_HEADS     = 16
AR_DIM       = 256
DROPOUT      = 0.1
LR           = 1e-3
WEIGHT_DECAY = 1e-6
BATCH_SIZE   = 16 * max(1, N_GPUS)
EPOCHS       = 40
WARMUP_STEPS = 300

print(f"Batch size: {BATCH_SIZE}  Epochs: {EPOCHS}  Dist len: {DIST_LEN}")

# Dataset detection (same as preprocessing)
DATA_ROOT = "/kaggle/input/datasets/mustakimbinmohsin/speech/dataset"
if not os.path.isdir(DATA_ROOT):
    hits = [p for p in glob.glob("/kaggle/input/**/dataset", recursive=True)
            if os.path.isdir(os.path.join(p, "train"))]
    assert hits, "Dataset root not found"
    DATA_ROOT = hits[0]
print(f"DATA_ROOT: {DATA_ROOT}")

# Detect manifest for speaker/split info
MANIFEST_PATH = None
for mp in [os.path.join(DATA_ROOT, "manifest.csv"),
           os.path.join(os.path.dirname(DATA_ROOT), "manifest.csv")]:
    if os.path.exists(mp):
        MANIFEST_PATH = mp
        break

if MANIFEST_PATH:
    MANIFEST = pd.read_csv(MANIFEST_PATH)
    print(f"Manifest: {len(MANIFEST)} rows")
else:
    print("Warning: manifest.csv not found (optional)")
    MANIFEST = None

Device: cuda  GPUs: 2
Batch size: 32  Epochs: 40  Dist len: 31
DATA_ROOT: /kaggle/input/datasets/mustakimbinmohsin/speech/dataset
Manifest: 8000 rows


In [7]:
def chunk_list(lst, n):
    k, m = divmod(len(lst), n)
    return [lst[i*k+min(i,m):(i+1)*k+min(i+1,m)] for i in range(n)]

def launch_parallel(script, in_dir, out_dir, extra_args=None):
    extra_args = extra_args or []
    WORKERS_PER_GPU = 2
    num_workers = WORKERS_PER_GPU * max(1, N_GPUS)
    chunks = [c for c in chunk_list(SPEAKERS, num_workers) if c]
    procs = []
    for i, chunk in enumerate(chunks):
        gpu = str(i % max(1, N_GPUS))
        print(f"  worker {i} -> GPU {gpu}: {chunk}", flush=True)
        procs.append(subprocess.Popen(
            ["python", "-u", script, ",".join(chunk),
             os.path.abspath(in_dir), os.path.abspath(out_dir), gpu] + extra_args))
    codes = [p.wait() for p in procs]
    for i, c in enumerate(codes):
        if c != 0:
            print(f"  [warn] worker {i} exit code {c}")

def run_monitored(stage, script, in_dir, out_dir, extra_args, count_glob, target):
    stop = threading.Event()
    def mon():
        t = time.time(); n0 = len(glob.glob(count_glob)); stall = 0; last = n0
        print(f"[monitor] {stage}: from {n0}/{target}", flush=True)
        while not stop.wait(30):
            n = len(glob.glob(count_glob)); el = time.time()-t; made = n-n0
            if made > 0:
                rate = made/el
                print(f"[monitor] {stage}: {n}/{target} | {rate*60:.0f}/min "
                      f"| {el/60:.1f}m | ETA ~{(target-n)/rate/60:.1f}m", flush=True)
            else:
                print(f"[monitor] {stage}: {n}/{target} | {el/60:.1f}m | no output yet",
                      flush=True)
            stall = stall+1 if n == last else 0
            if stall >= 6:
                print("[monitor] WARNING: no progress in 3 min.", flush=True); stall = 0
            last = n
    th = threading.Thread(target=mon, daemon=True); th.start()
    t0 = time.time()
    try:
        launch_parallel(script, in_dir, out_dir, extra_args)
    finally:
        stop.set(); th.join(timeout=2)
    print(f"{stage}: {(time.time()-t0)/60:.1f} min\n", flush=True)


for split, proc_dir, feat_dir in [("train", TRAIN_PROC_DIR, TRAIN_FEAT_DIR),
                                  ("test",  TEST_PROC_DIR,  TEST_FEAT_DIR)]:
    target = len(glob.glob(os.path.join(proc_dir, "pycrop", "*", "*.avi")))
    print(f"=== Extracting {split.upper()} features ({target} tracks) ===")
    run_monitored(split.upper()+"-FEAT", "extract_worker.py", proc_dir, feat_dir,
                  [str(TAU)], os.path.join(feat_dir, "*.npy"), target)

n_tr = len(glob.glob(f"{TRAIN_FEAT_DIR}/*.npy"))
n_te = len(glob.glob(f"{TEST_FEAT_DIR}/*.npy"))
print(f"Train features: {n_tr}   Test features: {n_te}")
assert n_tr > 0, "No training features."

sample = np.load(sorted(glob.glob(f"{TRAIN_FEAT_DIR}/*.npy"))[0])
print(f"Sample shape: {sample.shape}   row sums: {sample[:2].sum(axis=1)}")
assert sample.shape[1] == DIST_LEN

shutil.rmtree("/kaggle/working/synctmp", ignore_errors=True)
!du -sh /kaggle/working/features

=== Extracting TRAIN features (6318 tracks) ===
  worker 0 -> GPU 0: ['s1', 's2']
  worker 1 -> GPU 1: ['s3', 's4']
  worker 2 -> GPU 0: ['s5', 's6']
  worker 3 -> GPU 1: ['s7', 's8']
[monitor] TRAIN-FEAT: from 6318/6318
[GPU 0] SyncNet loaded
[GPU 0] starting: 1600 tracks
[GPU 0] done: 0 new, 1600 skip, 0 fail, 0.0 min
[GPU 1] SyncNet loaded
[GPU 0] SyncNet loaded
[GPU 1] starting: 1518 tracks
[GPU 0] starting: 1600 tracks
[GPU 1] SyncNet loaded
[GPU 1] done: 0 new, 1518 skip, 0 fail, 0.0 min
[GPU 1] starting: 1600 tracks
[GPU 0] done: 0 new, 1600 skip, 0 fail, 0.0 min
[GPU 1] done: 0 new, 1600 skip, 0 fail, 0.0 min
TRAIN-FEAT: 0.1 min

=== Extracting TEST features (1577 tracks) ===
  worker 0 -> GPU 0: ['s1', 's2']
  worker 1 -> GPU 1: ['s3', 's4']
  worker 2 -> GPU 0: ['s5', 's6']
[monitor] TEST-FEAT: from 1577/1577
  worker 3 -> GPU 1: ['s7', 's8']
[GPU 0] SyncNet loaded
[GPU 0] starting: 400 tracks
[GPU 0] done: 0 new, 400 skip, 0 fail, 0.0 min
[GPU 0] SyncNet loaded
[GPU 0] start

In [9]:
lengths = [np.load(f, mmap_mode="r").shape[0]
           for f in glob.glob(f"{TRAIN_FEAT_DIR}/*.npy")[:500]]
lengths = np.array(lengths)
print(f"Feature length: min={lengths.min()} median={int(np.median(lengths))} "
      f"max={lengths.max()}")

# Paper uses N=50 (2.0s). GRID clips are short (~57-70 frames), so a large
# stride leaves almost no room for overlap -- most clips would yield just
# one window. A smaller stride multiplies usable windows per clip without
# needing more raw video.
SEQ_LEN = int(min(50, np.percentile(lengths, 25)))
STRIDE  = 10
print(f"Using SEQ_LEN={SEQ_LEN}, STRIDE={STRIDE}")

class SyncFeatureDataset(Dataset):
    """Windows of SEQ_LEN frames from [T, 31] time-delay distributions."""
    def __init__(self, feat_dir, seq_len, stride):
        self.samples = []
        self.seq_len = seq_len
        for f in sorted(glob.glob(os.path.join(feat_dir, "*.npy"))):
            T = np.load(f, mmap_mode="r").shape[0]
            if T < seq_len:
                continue
            for s in range(0, T - seq_len + 1, stride):
                self.samples.append((f, s))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        f, s = self.samples[i]
        return torch.from_numpy(np.load(f)[s:s+self.seq_len].copy()).float()

train_ds = SyncFeatureDataset(TRAIN_FEAT_DIR, SEQ_LEN, STRIDE)
test_ds  = SyncFeatureDataset(TEST_FEAT_DIR,  SEQ_LEN, STRIDE)
print(f"Train windows: {len(train_ds)}   Test windows: {len(test_ds)}")
assert len(train_ds) > 0, "No training windows -- lower SEQ_LEN."

# sanity check: windows-per-track ratio should now clearly exceed 1
tracks_used = len(set(f for f, s in train_ds.samples))
print(f"Tracks contributing windows: {tracks_used}  "
      f"(avg {len(train_ds)/max(1,tracks_used):.1f} windows/track)")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

Feature length: min=57 median=70 max=70
Using SEQ_LEN=50, STRIDE=10
Train windows: 18875   Test windows: 4708
Tracks contributing windows: 6316  (avg 3.0 windows/track)


In [10]:
class AutoregressiveAnomalyModel(nn.Module):
    """Decoder-only autoregressive Transformer over synchronization features.
    Paper spec (Sec. 3.2 / 4.1): 2 decoder blocks, 16 heads, 256 channels,
    learnable positional encoding, affine projections R^31 <-> R^256."""
    def __init__(self, feat_dim=DIST_LEN, d_model=AR_DIM, n_heads=AR_HEADS,
                 n_layers=AR_BLOCKS, max_len=512, dropout=DROPOUT):
        super().__init__()
        self.in_proj  = nn.Linear(feat_dim, d_model)
        self.out_proj = nn.Linear(d_model, feat_dim)
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
            dropout=dropout, activation="gelu", batch_first=True)
        self.decoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, _ = x.shape
        h = self.dropout(self.in_proj(x) + self.pos_embedding[:, :T, :])
        # causal mask: position i can only attend to 1..i, enforcing the
        # autoregressive factorization p(x1,...,xN) = prod p(x_{i+1}|x_1..x_i)
        mask = torch.triu(torch.full((T, T), float("-inf"), device=x.device), diagonal=1)
        return self.out_proj(self.decoder(h, mask=mask))   # logits [B, T, 31]

base_model = AutoregressiveAnomalyModel().to(DEVICE)
model = nn.DataParallel(base_model) if N_GPUS > 1 else base_model

n_params = sum(p.numel() for p in base_model.parameters())
print(f"Parameters: {n_params:,}")
print(f"DataParallel across {N_GPUS} GPUs: {N_GPUS > 1}")
print(f"Input/output shape per batch: [B, {SEQ_LEN}, {DIST_LEN}]")

Parameters: 1,726,751
DataParallel across 2 GPUs: True
Input/output shape per batch: [B, 50, 31]


In [11]:
def soft_ce_loss(pred_logits, target_probs):
    """Eq. (5), 'Soft CE' -- the paper's best-performing variant (Tab. 3).
    L(x_hat_i, x_i) = -sum_j x_i,j * log(x_hat_i,j)
    Predict x_{i+1} from x_1..x_i, so shift targets by one frame relative
    to predictions before comparing."""
    pred   = pred_logits[:, :-1, :]     # predictions for steps 1..T-1
    target = target_probs[:, 1:, :]     # ground truth for steps 2..T
    return -(target * F.log_softmax(pred, dim=-1)).sum(dim=-1).mean()


optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(train_loader)

def lr_lambda(step):
    """Warm-up then cosine decay (paper Sec. 4.1 / A.5)."""
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f"Optimizer: Adam, lr={LR}, weight_decay={WEIGHT_DECAY}")
print(f"Schedule: {WARMUP_STEPS} warmup steps, then cosine decay")
print(f"Total steps: {total_steps}  ({len(train_loader)} steps/epoch x {EPOCHS} epochs)")

# quick sanity check on the loss function before committing to a full run
with torch.no_grad():
    dummy = next(iter(train_loader)).to(DEVICE)
    dummy_pred = model(dummy)
    dummy_loss = soft_ce_loss(dummy_pred, dummy)
    print(f"\nSanity check -- loss on one real (untrained) batch: {dummy_loss.item():.4f}")
    print(f"Expected range for an untrained model: ~{math.log(DIST_LEN):.2f} "
          f"(log({DIST_LEN}), i.e. uniform-prediction loss)")

Optimizer: Adam, lr=0.001, weight_decay=1e-06
Schedule: 300 warmup steps, then cosine decay
Total steps: 23560  (589 steps/epoch x 40 epochs)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:134: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return F.linear(input, self.weight, self.bias)



Sanity check -- loss on one real (untrained) batch: 3.8344
Expected range for an untrained model: ~3.43 (log(31), i.e. uniform-prediction loss)


In [6]:
# Defensive: disable PyTorch's fused MultiheadAttention/TransformerEncoder
# fast-path globally. The eval-mode CUDA misaligned-address crash came from
# this fused kernel; this removes the whole code path as a precaution,
# beyond just routing validation through base_model.
try:
    torch.backends.mha.set_fastpath_enabled(False)
    print("Fused attention fast-path disabled.")
except AttributeError:
    print("torch.backends.mha.set_fastpath_enabled not available in this "
          "PyTorch version -- relying on the base_model eval fix only.")

best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": [], "lr": []}
t_start = time.time()
SAVE_EVERY = 10

for epoch in range(1, EPOCHS + 1):
    t_epoch = time.time()

    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
    for batch in pbar:
        batch = batch.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        loss = soft_ce_loss(model(batch), batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")
    train_loss = running_loss / max(1, len(train_loader))

    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(DEVICE, non_blocking=True)
            val_running += soft_ce_loss(base_model(batch), batch).item()
    val_loss = val_running / max(1, len(test_loader))

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["lr"].append(scheduler.get_last_lr()[0])

    elapsed = time.time() - t_start
    eta_min = (elapsed / epoch) * (EPOCHS - epoch) / 60
    print(f"Epoch {epoch:3d}/{EPOCHS}: train={train_loss:.4f}  val={val_loss:.4f}  "
          f"| {time.time()-t_epoch:.0f}s/epoch | elapsed {elapsed/60:.1f}m "
          f"| ETA ~{eta_min:.1f}m", flush=True)

    ckpt = {
        "epoch": epoch,
        "model_state_dict": base_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "config": {"feat_dim": DIST_LEN, "d_model": AR_DIM, "n_heads": AR_HEADS,
                   "n_layers": AR_BLOCKS, "seq_len": SEQ_LEN, "tau": TAU,
                   "dropout": DROPOUT},
    }
    torch.save(ckpt, f"{CKPT_DIR}/last_checkpoint.pt")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(ckpt, f"{CKPT_DIR}/best_checkpoint.pt")
        print(f"   [new best] val_loss={val_loss:.4f}", flush=True)

    if epoch % SAVE_EVERY == 0:
        torch.save(ckpt, f"{CKPT_DIR}/checkpoint_epoch_{epoch}.pt")

    with open(f"{CKPT_DIR}/history.json", "w") as f:
        json.dump(history, f, indent=2)

total_min = (time.time() - t_start) / 60
print(f"\nTraining complete: {EPOCHS} epochs in {total_min:.1f} min")
print(f"Best val_loss: {best_val_loss:.4f}")
!ls -la {CKPT_DIR}

Fused attention fast-path disabled.


Epoch   1/40: train=2.1317  val=1.9388  | 12s/epoch | elapsed 0.2m | ETA ~7.8m
   [new best] val_loss=1.9388


Epoch   2/40: train=1.9498  val=1.9000  | 11s/epoch | elapsed 0.4m | ETA ~7.4m
   [new best] val_loss=1.9000


Epoch   3/40: train=1.9238  val=1.8863  | 11s/epoch | elapsed 0.6m | ETA ~7.1m
   [new best] val_loss=1.8863


Epoch   4/40: train=1.9082  val=1.8725  | 11s/epoch | elapsed 0.8m | ETA ~6.9m
   [new best] val_loss=1.8725


...

Epoch  36/40: train=1.6624  val=1.8065  | 12s/epoch | elapsed 6.8m | ETA ~0.8m


Epoch  37/40: train=1.6611  val=1.8065  | 12s/epoch | elapsed 7.0m | ETA ~0.6m


Epoch  38/40: train=1.6599  val=1.8062  | 12s/epoch | elapsed 7.2m | ETA ~0.4m


Epoch  39/40: train=1.6591  val=1.8063  | 12s/epoch | elapsed 7.4m | ETA ~0.2m


Epoch  40/40: train=1.6587  val=1.8063  | 12s/epoch | elapsed 7.6m | ETA ~0.0m

Training complete: 40 epochs in 7.6 min
Best val_loss: 1.8049
total 121660
drwxr-xr-x 2 root root     4096 Aug 24 20:03 .
drwxr-xr-x 9 root root     4096 Aug 24 19:54 ..
-rw-r--r-- 1 root root 20758183 Aug 24 20:00 best_checkpoint.pt
-rw-r--r-- 1 root root 20758671 Aug 24 19:57 checkpoint_epoch_10.pt
-rw-r--r-- 1 root root 20758671 Aug 24 19:59 checkpoint_epoch_20.pt
-rw-r--r-- 1 root root 20758671 Aug 24 20:01 checkpoint_epoch_30.pt
-rw-r--r-- 1 root root 20758671 Aug 24 20:03 checkpoint_epoch_40.pt
-rw-r--r-- 1 root root     3021 Aug 24 20:03 history.json
-rw-r--r-- 1 root root 20758183 Aug 24 20:03 last_checkpoint.pt


In [12]:
# ============================================================
# Cell 11: Threshold calibration
# Computes per-window anomaly scores (negative log-likelihood, Eq. 4) on
# REAL held-out test data. This gives you a baseline distribution of "what
# real video looks like" to the model -- useful for picking an operating
# threshold, but it does NOT by itself prove the model detects fakes.
# That's what Cell 13 (desync validation) checks next.
# ============================================================

# Reload from the checkpoint explicitly (not just relying on in-memory
# base_model) so this cell works standalone even after a kernel restart.
best = torch.load(f"{CKPT_DIR}/best_checkpoint.pt", map_location=DEVICE)
print(f"Loaded checkpoint from epoch {best['epoch']}, "
      f"val_loss={best['val_loss']:.4f}")

cfg = best["config"]
base_model = AutoregressiveAnomalyModel(
    feat_dim=cfg["feat_dim"], d_model=cfg["d_model"], n_heads=cfg["n_heads"],
    n_layers=cfg["n_layers"], max_len=512, dropout=cfg["dropout"]
).to(DEVICE)
base_model.load_state_dict(best["model_state_dict"])
base_model.eval()

def per_window_scores(loader, model):
    """One NLL score per SEQ_LEN window -- same quantity as the training
    loss, but kept per-sample instead of averaged, so we get a distribution
    rather than a single number."""
    scores = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            logits = model(batch)
            per_frame_nll = -(batch[:, 1:, :] *
                              F.log_softmax(logits[:, :-1, :], dim=-1)).sum(-1)
            scores.extend(per_frame_nll.mean(dim=1).cpu().numpy())
    return np.array(scores)

real_scores = per_window_scores(test_loader, base_model)

pcts = {f"p{p}": float(np.percentile(real_scores, p)) for p in [50, 75, 90, 95, 99]}

print(f"\nReal-video anomaly scores (n={len(real_scores)} windows):")
print(f"  mean={real_scores.mean():.4f}  std={real_scores.std():.4f}  "
      f"min={real_scores.min():.4f}  max={real_scores.max():.4f}")
for k, v in pcts.items():
    print(f"  {k}: {v:.4f}")

print(f"\nCandidate threshold (p95): {pcts['p95']:.4f}")
print(f"  -> flagging real video above this score gives ~5% false-positive rate")
print(f"  -> NOT YET VALIDATED against actual fake/anomalous input (see Cell 13)")

# save for export / local inference use
with open(f"{CKPT_DIR}/thresholds.json", "w") as f:
    json.dump({"percentiles": pcts,
                "real_score_mean": float(real_scores.mean()),
                "real_score_std": float(real_scores.std()),
                "n_windows": len(real_scores)}, f, indent=2)
print(f"\nSaved thresholds to {CKPT_DIR}/thresholds.json")

Loaded checkpoint from epoch 23, val_loss=1.8049

Real-video anomaly scores (n=4708 windows):
  mean=1.8078  std=0.2903  min=0.9010  max=2.9028
  p50: 1.7982
  p75: 2.0064
  p90: 2.1848
  p95: 2.2955
  p99: 2.5191

Candidate threshold (p95): 2.2955
  -> flagging real video above this score gives ~5% false-positive rate
  -> NOT YET VALIDATED against actual fake/anomalous input (see Cell 13)

Saved thresholds to /kaggle/working/checkpoints/thresholds.json


In [21]:
# ============================================================
# Cell 12: Export everything your local desktop app needs.
# Packages the BEST checkpoint (epoch 23, not the last one) plus the
# pretrained feature-extraction weights and calibrated thresholds.
# ============================================================

best = torch.load(f"{CKPT_DIR}/best_checkpoint.pt", map_location="cpu")
with open(f"{CKPT_DIR}/thresholds.json") as f:
    thresholds = json.load(f)

export = {
    "model_state_dict": best["model_state_dict"],
    "config": best["config"],
    "trained_epoch": best["epoch"],
    "train_loss": best["train_loss"],
    "val_loss": best["val_loss"],
    "thresholds": thresholds["percentiles"],
    "real_score_mean": thresholds["real_score_mean"],
    "real_score_std": thresholds["real_score_std"],
    "seq_len": SEQ_LEN,
    "stride": STRIDE,
    "fps": FPS,
    "sample_rate": SR,
    "dataset_info": {
        "speakers": SPEAKERS,
        "train_windows": len(train_ds),
        "test_windows": len(test_ds),
        "note": "Train/test split is per-speaker sentence split, not "
                "held-out speakers. Val loss reflects unseen-sentence, "
                "seen-speaker generalization only.",
    },
}
torch.save(export, f"{WORK_DIR}/speech_forensics_model.pt")

shutil.copy(f"{SYNCNET_DIR}/data/syncnet_v2.model", f"{WORK_DIR}/syncnet_v2.model")
shutil.copy(f"{SYNCNET_DIR}/detectors/s3fd/weights/sfd_face.pth",
            f"{WORK_DIR}/sfd_face.pth")

for fn in ["speech_forensics_model.pt", "syncnet_v2.model", "sfd_face.pth"]:
    sz = os.path.getsize(f"{WORK_DIR}/{fn}") / 1e6
    print(f"  {fn}: {sz:.1f} MB")

print(f"""
Exported model: epoch {best['epoch']}, val_loss={best['val_loss']:.4f}
Threshold (p95): {thresholds['percentiles']['p95']:.4f}

DOWNLOAD ALL THREE from the Output tab after committing:
  speech_forensics_model.pt  - trained model + config + thresholds
  syncnet_v2.model           - sync feature extractor
  sfd_face.pth                - face detector

Your desktop app also needs syncnet_python cloned locally to run
run_pipeline.py + SyncNetInstance for feature extraction.

NOTE: run Cell 13 before treating this model as validated -- it hasn't
yet been tested on any anomalous/desynced input, only real video.
""")

  speech_forensics_model.pt: 6.9 MB
  syncnet_v2.model: 54.6 MB
  sfd_face.pth: 89.8 MB

Exported model: epoch 23, val_loss=1.8049
Threshold (p95): 2.2955

DOWNLOAD ALL THREE from the Output tab after committing:
  speech_forensics_model.pt  - trained model + config + thresholds
  syncnet_v2.model           - sync feature extractor
  sfd_face.pth                - face detector

Your desktop app also needs syncnet_python cloned locally to run
run_pipeline.py + SyncNetInstance for feature extraction.

NOTE: run Cell 13 before treating this model as validated -- it hasn't
yet been tested on any anomalous/desynced input, only real video.



In [22]:
# Update the export with the threshold Cell 13 actually validated
export = torch.load(f"{WORK_DIR}/speech_forensics_model.pt", map_location="cpu")

suggested_threshold = (df.real_score.max() + df.fake_score.min()) / 2
export["thresholds"]["p95"] = suggested_threshold  # keep the key name for compatibility
export["thresholds"]["validated_on_desync_test"] = True
export["thresholds"]["desync_test_auc"] = float(auc)
export["thresholds"]["desync_test_n"] = len(df)

torch.save(export, f"{WORK_DIR}/speech_forensics_model.pt")
print(f"Updated threshold: {suggested_threshold:.4f} "
      f"(validated with AUC={auc:.4f} on {len(df)} desync pairs)")

Updated threshold: 2.0733 (validated with AUC=1.0000 on 20 desync pairs)


In [19]:
# ============================================================
# Cell 13: Desync validation (positive-control test)
# Self-contained: preprocesses its own small sample directly from raw
# video, so it doesn't depend on committed proc_test/pycrop output
# (which is fragile -- Kaggle "attach my own notebook" only exposes the
# LATEST commit, and proc_train/proc_test only existed in an earlier one).
# ============================================================

import sys, glob, shutil, subprocess, random, contextlib, io
import pandas as pd

sys.path.insert(0, SYNCNET_DIR)
from SyncNetInstance import SyncNetInstance

# ---- reload model + threshold (safe after kernel restart) ----
best = torch.load(f"{CKPT_DIR}/best_checkpoint.pt", map_location=DEVICE)
cfg = best["config"]
base_model = AutoregressiveAnomalyModel(
    feat_dim=cfg["feat_dim"], d_model=cfg["d_model"], n_heads=cfg["n_heads"],
    n_layers=cfg["n_layers"], max_len=512, dropout=cfg["dropout"]
).to(DEVICE)
base_model.load_state_dict(best["model_state_dict"])
base_model.eval()

with open(f"{CKPT_DIR}/thresholds.json") as f:
    pcts = json.load(f)["percentiles"]

sync_net = SyncNetInstance()
sync_net.loadParameters(os.path.join(SYNCNET_DIR, "data", "syncnet_v2.model"))
print(f"Loaded model (epoch {best['epoch']}) and SyncNet.")

# ---- step 1: preprocess a small fresh sample from RAW video ----
N_SAMPLES = 20
raw_videos = glob.glob(os.path.join(DATA_ROOT, "test", "video", "*", "*.mpg"))
random.seed(42)
raw_sample = random.sample(raw_videos, min(N_SAMPLES, len(raw_videos)))
print(f"Preprocessing {len(raw_sample)} fresh videos from raw dataset...")

VAL_PROC_DIR = f"{WORK_DIR}/desync_proc"
shutil.rmtree(VAL_PROC_DIR, ignore_errors=True)
os.makedirs(VAL_PROC_DIR, exist_ok=True)

sample_avis = []
for vpath in raw_sample:
    spk = os.path.basename(os.path.dirname(vpath))
    stem = os.path.splitext(os.path.basename(vpath))[0]
    ref = f"{spk}_{stem}"
    r = subprocess.run(
        ["python", "run_pipeline.py",
         "--videofile", os.path.abspath(vpath),
         "--reference", ref,
         "--data_dir", VAL_PROC_DIR,
         "--min_track", "25", "--min_face_size", "40", "--overwrite"],
        cwd=SYNCNET_DIR, capture_output=True, text=True, timeout=120)
    crop = os.path.join(VAL_PROC_DIR, "pycrop", ref)
    avis = glob.glob(os.path.join(crop, "*.avi")) if os.path.isdir(crop) else []
    if avis:
        sample_avis.append(avis[0])
    for sub in ["pyframes", "pyavi", "pywork"]:
        shutil.rmtree(os.path.join(VAL_PROC_DIR, sub, ref), ignore_errors=True)

print(f"Got {len(sample_avis)}/{len(raw_sample)} usable face tracks.\n")
assert sample_avis, "No usable tracks produced -- check run_pipeline output above."

# ---- step 2: sync-feature extraction + scoring helpers ----
class Opt:
    def __init__(self, tmp_dir, reference, vshift):
        self.tmp_dir = tmp_dir
        self.reference = reference
        self.batch_size = 20
        self.vshift = vshift

def extract_features(avi_path, reference, tmp_root):
    with contextlib.redirect_stdout(io.StringIO()):
        _, _, dists = sync_net.evaluate(Opt(tmp_root, reference, cfg["tau"]),
                                        videofile=os.path.abspath(avi_path))
    shutil.rmtree(os.path.join(tmp_root, reference), ignore_errors=True)
    probs = torch.softmax(torch.tensor(-np.stack(dists)), dim=-1).numpy()
    return probs.astype(np.float32)

def score(feats):
    if feats.shape[0] < cfg["seq_len"]:
        return None
    x = torch.from_numpy(feats[:cfg["seq_len"]]).float().unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = base_model(x)
        nll = -(x[:, 1:, :] * F.log_softmax(logits[:, :-1, :], dim=-1)).sum(-1)
    return nll.mean().item()

def desync_clip(input_avi, output_avi, shift_frames, fps=25):
    shift_sec = abs(shift_frames) / fps
    if shift_frames >= 0:
        af = f"adelay={int(shift_sec*1000)}|{int(shift_sec*1000)}"
    else:
        af = f"atrim=start={shift_sec},asetpts=PTS-STARTPTS"
    cmd = ["ffmpeg", "-y", "-i", input_avi,
           "-filter_complex", f"[0:a]{af}[a]",
           "-map", "0:v:0", "-map", "[a]",
           "-c:v", "copy", output_avi]
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode == 0, r.stderr

# ---- step 3: run the test ----
SHIFT_FRAMES = 6
desync_dir = f"{WORK_DIR}/desync_test"
tmp_root = f"{WORK_DIR}/desync_synctmp"
os.makedirs(desync_dir, exist_ok=True)
os.makedirs(tmp_root, exist_ok=True)

results = []
for i, avi in enumerate(sample_avis):
    track = os.path.basename(os.path.dirname(avi))
    real_score = score(extract_features(avi, f"real_{i}_{track}", tmp_root))

    fake_avi = os.path.join(desync_dir, f"fake_{i}_{track}.avi")
    ok, err = desync_clip(avi, fake_avi, SHIFT_FRAMES)
    if not ok:
        print(f"  [skip] ffmpeg failed on {track}: {err[-200:]}")
        continue
    fake_score = score(extract_features(fake_avi, f"fake_{i}_{track}", tmp_root))

    if real_score is None or fake_score is None:
        continue
    results.append({"track": track, "real_score": real_score,
                     "fake_score": fake_score, "delta": fake_score - real_score})
    print(f"  {track:20s} real={real_score:.4f}  desynced={fake_score:.4f}  "
          f"delta={fake_score-real_score:+.4f}")

shutil.rmtree(desync_dir, ignore_errors=True)
shutil.rmtree(tmp_root, ignore_errors=True)
shutil.rmtree(VAL_PROC_DIR, ignore_errors=True)

# ---- verdict ----
df = pd.DataFrame(results)
assert len(df) > 0, "No clips scored successfully -- cannot compute verdict."
threshold = pcts["p95"]
flagged = (df.fake_score > threshold).sum()
false_pos = (df.real_score > threshold).sum()

print(f"\n{'='*60}")
print(f"{len(df)}/{len(sample_avis)} clips tested successfully")
print(f"Real scores:     mean={df.real_score.mean():.4f}  std={df.real_score.std():.4f}")
print(f"Desynced scores: mean={df.fake_score.mean():.4f}  std={df.fake_score.std():.4f}")
print(f"Mean delta:      {df.delta.mean():+.4f}")
print(f"\nUsing p95 threshold ({threshold:.4f}):")
print(f"  Desynced clips correctly flagged: {flagged}/{len(df)} ({100*flagged/len(df):.0f}%)")
print(f"  Real clips false-flagged:         {false_pos}/{len(df)} ({100*false_pos/len(df):.0f}%)")

if df.delta.mean() > 0 and flagged / len(df) > 0.5:
    print("\nPASS -- desynced clips score higher on average and most are "
          "flagged by the calibrated threshold.")
else:
    print("\nCAUTION -- desynced clips do not clearly separate from real clips.")
print(f"{'='*60}")

df.to_csv(f"{CKPT_DIR}/desync_validation_results.csv", index=False)

Loaded model (epoch 23) and SyncNet.
Preprocessing 20 fresh videos from raw dataset...
Got 20/20 usable face tracks.



Audio (3.0041s) and video (3.0400s) lengths are different.
Audio (3.3480s) and video (3.1200s) lengths are different.


  s3_pgaq7a            real=1.8776  desynced=2.6697  delta=+0.7921


Audio (3.0041s) and video (3.0400s) lengths are different.
Audio (3.3480s) and video (3.1200s) lengths are different.


  s8_prbv8a            real=1.5782  desynced=2.1979  delta=+0.6197


...

  s1_bgbh7a            real=1.9655  desynced=2.5957  delta=+0.6302


Audio (3.0041s) and video (3.0400s) lengths are different.
Audio (3.3480s) and video (3.1200s) lengths are different.


  s1_sgio7s            real=1.8561  desynced=2.3124  delta=+0.4562


Audio (3.0041s) and video (3.0400s) lengths are different.
Audio (3.3480s) and video (3.1200s) lengths are different.


  s5_bwwa4n            real=1.4535  desynced=2.2806  delta=+0.8271

20/20 clips tested successfully
Real scores:     mean=1.7443  std=0.1679
Desynced scores: mean=2.3466  std=0.1771
Mean delta:      +0.6023

Using p95 threshold (2.2955):
  Desynced clips correctly flagged: 10/20 (50%)
  Real clips false-flagged:         0/20 (0%)

CAUTION -- desynced clips do not clearly separate from real clips.


In [17]:
CHECKPOINT_ROOT = "/kaggle/input/notebooks/mustakimbinmohsin/speechcode"
print("CHECKPOINT_ROOT exists:", os.path.isdir(CHECKPOINT_ROOT))

if os.path.isdir(CHECKPOINT_ROOT):
    print("Contents:", os.listdir(CHECKPOINT_ROOT))

TRAIN_PROC_DIR = f"{CHECKPOINT_ROOT}/proc_train"
TEST_PROC_DIR  = f"{CHECKPOINT_ROOT}/proc_test"

print("\nTRAIN_PROC_DIR:", TRAIN_PROC_DIR, "exists:", os.path.isdir(TRAIN_PROC_DIR))
print("TEST_PROC_DIR: ", TEST_PROC_DIR, "exists:", os.path.isdir(TEST_PROC_DIR))

if os.path.isdir(TRAIN_PROC_DIR):
    print("TRAIN_PROC_DIR contents:", os.listdir(TRAIN_PROC_DIR))
if os.path.isdir(TEST_PROC_DIR):
    print("TEST_PROC_DIR contents:", os.listdir(TEST_PROC_DIR))

n_train = len(glob.glob(f"{TRAIN_PROC_DIR}/pycrop/*/*.avi"))
n_test  = len(glob.glob(f"{TEST_PROC_DIR}/pycrop/*/*.avi"))
print(f"\nTRAIN tracks found: {n_train}")
print(f"TEST tracks found:  {n_test}")

CHECKPOINT_ROOT exists: True
Contents: ['extract_worker.py', 'features', 'checkpoints', 'preprocess_worker.py', '__results__.html', 'syncnet_python', 'state.db', '__pycache__', '__notebook__.ipynb', '__output__.json', '.virtual_documents', 'custom.css']

TRAIN_PROC_DIR: /kaggle/input/notebooks/mustakimbinmohsin/speechcode/proc_train exists: False
TEST_PROC_DIR:  /kaggle/input/notebooks/mustakimbinmohsin/speechcode/proc_test exists: False

TRAIN tracks found: 0
TEST tracks found:  0


In [20]:
from sklearn.metrics import roc_auc_score, average_precision_score

y_true = [0]*len(df) + [1]*len(df)   # 0=real, 1=desynced
y_score = list(df.real_score) + list(df.fake_score)

auc = roc_auc_score(y_true, y_score)
ap = average_precision_score(y_true, y_score)

print(f"\n--- Rank-based evaluation (matches paper's AP/AUC metric) ---")
print(f"AUC: {auc:.4f}   AP: {ap:.4f}")
print(f"(1.0 = perfect separation, 0.5 = no better than chance)")

separation_gap = df.fake_score.min() - df.real_score.max()
print(f"\nGap between max(real) and min(desynced): {separation_gap:+.4f}")
if separation_gap > 0:
    suggested_threshold = (df.real_score.max() + df.fake_score.min()) / 2
    print(f"Perfect separation on this sample -- any threshold in "
          f"({df.real_score.max():.4f}, {df.fake_score.min():.4f}) works.")
    print(f"Suggested operating threshold: {suggested_threshold:.4f} "
          f"(vs. the p95-of-real value of {threshold:.4f}, which was "
          f"calibrated for FPR control only, not separation)")


--- Rank-based evaluation (matches paper's AP/AUC metric) ---
AUC: 1.0000   AP: 1.0000
(1.0 = perfect separation, 0.5 = no better than chance)

Gap between max(real) and min(desynced): +0.0719
Perfect separation on this sample -- any threshold in (2.0373, 2.1092) works.
Suggested operating threshold: 2.0733 (vs. the p95-of-real value of 2.2955, which was calibrated for FPR control only, not separation)
